In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
df = pd.read_csv("../data/processed/cleaned_insurance_data.csv")
df.head()

,months_as_customer,age,policy_number,policy_bind_date,policy_state,policy_csl,policy_deductable,policy_annual_premium,umbrella_limit,insured_zip,insured_sex,insured_education_level,insured_occupation,insured_hobbies,insured_relationship,capital-gains,capital-loss,incident_date,incident_type,collision_type,incident_severity,authorities_contacted,incident_state,incident_city,incident_location,incident_hour_of_the_day,number_of_vehicles_involved,property_damage,bodily_injuries,witnesses,police_report_available,total_claim_amount,injury_claim,property_claim,vehicle_claim,auto_make,auto_model,auto_year,fraud_reported,policy_duration_days,claim_to_premium_ratio
0,328,48,521585,2014-10-17,OH,250/500,1000,1406.91,0,466132,MALE,MD,craft-repair,sleeping,husband,53300,0,2015-01-25,Single Vehicle Collision,Side Collision,Major Damage,Police,SC,Columbus,9935 4th Drive,5,1,YES,1,2,YES,71610,6510,13020,52080,Saab,92x,2004,1,100,50.898778
1,228,42,342868,2006-06-27,IN,250/500,2000,1197.22,5000000,468176,MALE,MD,machine-op-inspct,reading,other-relative,0,0,2015-01-21,Vehicle Theft,NaN,Minor Damage,Police,VA,Riverwood,6608 MLK Hwy,8,1,NaN,0,0,NaN,5070,780,780,3510,Mercedes,E400,2007,1,3130,4.234811
2,134,29,687698,2000-09-06,OH,100/300,2000,1413.14,5000000,430632,FEMALE,PhD,sales,board-games,own-child,35100,0,2015-02-22,Multi-vehicle Collision,Rear Collision,Minor Damage,Police,NY,Columbus,7121 Francis Lane,7,3,NO,2,3,NO,34650,7700,3850,23100,Dodge,RAM,2007,0,5282,24.519864
3,256,41,227811,1990-05-25,IL,250/500,2000,1415.74,6000000,608117,FEMALE,PhD,armed-forces,board-games,unmarried,48900,-62400,2015-01-10,Single Vehicle Collision,Front Collision,Major Damage,Police,OH,Arlington,6956 Maple Drive,5,1,NaN,1,2,NO,63400,6340,6340,50720,Chevrolet,Tahoe,2014,1,8996,44.782234
4,228,44,367455,2014-06-06,IL,500/1000,1000,1583.91,6000000,610706,MALE,Associate,sales,board-games,unmarried,66000,-46000,2015-02-17,Vehicle Theft,NaN,Minor Damage,NaN,NY,Arlington,3041 3rd Ave,20,1,NO,0,1,NO,6500,1300,650,4550,Accura,RSX,2009,0,256,4.103769


In [3]:
print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())
print("\nData types:\n")
print(df.dtypes)

Shape: (1000, 41)

Columns:
 ['months_as_customer', 'age', 'policy_number', 'policy_bind_date', 'policy_state', 'policy_csl', 'policy_deductable', 'policy_annual_premium', 'umbrella_limit', 'insured_zip', 'insured_sex', 'insured_education_level', 'insured_occupation', 'insured_hobbies', 'insured_relationship', 'capital-gains', 'capital-loss', 'incident_date', 'incident_type', 'collision_type', 'incident_severity', 'authorities_contacted', 'incident_state', 'incident_city', 'incident_location', 'incident_hour_of_the_day', 'number_of_vehicles_involved', 'property_damage', 'bodily_injuries', 'witnesses', 'police_report_available', 'total_claim_amount', 'injury_claim', 'property_claim', 'vehicle_claim', 'auto_make', 'auto_model', 'auto_year', 'fraud_reported', 'policy_duration_days', 'claim_to_premium_ratio']

Data types:

months_as_customer               int64
age                              int64
policy_number                    int64
policy_bind_date                object
policy_state 

In [4]:
df.columns = df.columns.str.strip()
df = df.replace("?", np.nan)

In [5]:
if df["fraud_reported"].dtype == "object":
    df["fraud_reported"] = df["fraud_reported"].map({"Y": 1, "N": 0})

df["fraud_reported"].value_counts(dropna=False)

fraud_reported
0    753
1    247
Name: count, dtype: int64

In [6]:
date_cols = ["policy_bind_date", "incident_date"]

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

df[date_cols].head()

,policy_bind_date,incident_date
0,2014-10-17,2015-01-25
1,2006-06-27,2015-01-21
2,2000-09-06,2015-02-22
3,1990-05-25,2015-01-10
4,2014-06-06,2015-02-17


In [7]:
if "policy_bind_date" in df.columns and "incident_date" in df.columns:
    df["policy_duration_days"] = (df["incident_date"] - df["policy_bind_date"]).dt.days

if "total_claim_amount" in df.columns and "policy_annual_premium" in df.columns:
    df["claim_to_premium_ratio"] = df["total_claim_amount"] / df["policy_annual_premium"]

In [8]:
if "injury_claim" in df.columns and "total_claim_amount" in df.columns:
    df["injury_claim_ratio"] = df["injury_claim"] / df["total_claim_amount"]

if "property_claim" in df.columns and "total_claim_amount" in df.columns:
    df["property_claim_ratio"] = df["property_claim"] / df["total_claim_amount"]

if "vehicle_claim" in df.columns and "total_claim_amount" in df.columns:
    df["vehicle_claim_ratio"] = df["vehicle_claim"] / df["total_claim_amount"]

if "incident_hour_of_the_day" in df.columns:
    df["is_night_incident"] = df["incident_hour_of_the_day"].apply(
        lambda x: 1 if pd.notnull(x) and (x < 6 or x > 22) else 0
    )

if "number_of_vehicles_involved" in df.columns:
    df["multi_vehicle_flag"] = df["number_of_vehicles_involved"].apply(
        lambda x: 1 if pd.notnull(x) and x > 1 else 0
    )

In [9]:
df = df.replace([np.inf, -np.inf], np.nan)

In [10]:
drop_cols = [
    "_c39",
    "policy_number",
    "policy_bind_date",
    "incident_date",
    "insured_zip",
    "incident_location"
]

existing_drop_cols = [col for col in drop_cols if col in df.columns]
df = df.drop(columns=existing_drop_cols)

print("Dropped columns:", existing_drop_cols)
print("New shape:", df.shape)

Dropped columns: ['policy_number', 'policy_bind_date', 'incident_date', 'insured_zip', 'incident_location']
New shape: (1000, 41)


In [11]:
missing = df.isnull().sum().sort_values(ascending=False)
missing[missing > 0]

property_damage            360
police_report_available    343
collision_type             178
authorities_contacted       91
dtype: int64

In [12]:
df.nunique().sort_values(ascending=False).head(20)

claim_to_premium_ratio      1000
policy_annual_premium        991
policy_duration_days         953
total_claim_amount           763
vehicle_claim                726
injury_claim                 638
property_claim               626
months_as_customer           391
capital-loss                 354
capital-gains                338
age                           46
auto_model                    39
incident_hour_of_the_day      24
vehicle_claim_ratio           23
auto_year                     21
insured_hobbies               20
property_claim_ratio          19
injury_claim_ratio            17
insured_occupation            14
auto_make                     14
dtype: int64

In [13]:
df.head()

,months_as_customer,age,policy_state,policy_csl,policy_deductable,policy_annual_premium,umbrella_limit,insured_sex,insured_education_level,insured_occupation,insured_hobbies,insured_relationship,capital-gains,capital-loss,incident_type,collision_type,incident_severity,authorities_contacted,incident_state,incident_city,incident_hour_of_the_day,number_of_vehicles_involved,property_damage,bodily_injuries,witnesses,police_report_available,total_claim_amount,injury_claim,property_claim,vehicle_claim,auto_make,auto_model,auto_year,fraud_reported,policy_duration_days,claim_to_premium_ratio,injury_claim_ratio,property_claim_ratio,vehicle_claim_ratio,is_night_incident,multi_vehicle_flag
0,328,48,OH,250/500,1000,1406.91,0,MALE,MD,craft-repair,sleeping,husband,53300,0,Single Vehicle Collision,Side Collision,Major Damage,Police,SC,Columbus,5,1,YES,1,2,YES,71610,6510,13020,52080,Saab,92x,2004,1,100,50.898778,0.090909,0.181818,0.727273,1,0
1,228,42,IN,250/500,2000,1197.22,5000000,MALE,MD,machine-op-inspct,reading,other-relative,0,0,Vehicle Theft,NaN,Minor Damage,Police,VA,Riverwood,8,1,NaN,0,0,NaN,5070,780,780,3510,Mercedes,E400,2007,1,3130,4.234811,0.153846,0.153846,0.692308,0,0
2,134,29,OH,100/300,2000,1413.14,5000000,FEMALE,PhD,sales,board-games,own-child,35100,0,Multi-vehicle Collision,Rear Collision,Minor Damage,Police,NY,Columbus,7,3,NO,2,3,NO,34650,7700,3850,23100,Dodge,RAM,2007,0,5282,24.519864,0.222222,0.111111,0.666667,0,1
3,256,41,IL,250/500,2000,1415.74,6000000,FEMALE,PhD,armed-forces,board-games,unmarried,48900,-62400,Single Vehicle Collision,Front Collision,Major Damage,Police,OH,Arlington,5,1,NaN,1,2,NO,63400,6340,6340,50720,Chevrolet,Tahoe,2014,1,8996,44.782234,0.100000,0.100000,0.800000,1,0
4,228,44,IL,500/1000,1000,1583.91,6000000,MALE,Associate,sales,board-games,unmarried,66000,-46000,Vehicle Theft,NaN,Minor Damage,NaN,NY,Arlington,20,1,NO,0,1,NO,6500,1300,650,4550,Accura,RSX,2009,0,256,4.103769,0.200000,0.100000,0.700000,0,0


In [14]:
df.to_csv("../data/processed/featured_insurance_data.csv", index=False)
print("Saved to ../data/processed/featured_insurance_data.csv")

Saved to ../data/processed/featured_insurance_data.csv
